# LD Clumping

This notebook performs linkage disequilibrium clumping on the HF-focused prioritized variant table using the LDlink SNPclip API. Variants are pruned based on pairwise r² in the 1000 Genomes EUR reference population (r² threshold = 0.1, MAF threshold = 0.01, GRCh38), retaining one independent representative per LD block. The output is a reduced table of independent HF-associated lead SNPs for downstream biological interpretation and eQTL annotation.

In [1]:
from pathlib import Path
import json
import time
import requests
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
hf_path  = Path("../data/final/prioritized_variants/hf_prioritized_variants.xlsx")

out_dir  = Path("../data/final/ld_clumping")
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / "hf_lead_snps.xlsx"

LDLINK_TOKEN  = "eb615941e0fe"
LDLINK_URL    = "https://ldlink.nih.gov/LDlinkRest/snpclip"
POPULATION    = "CEU+TSI+FIN+GBR+IBS"
GENOME_BUILD  = "grch38"
R2_THRESHOLD  = 0.1
MAF_THRESHOLD = 0.01
BATCH_SIZE    = 300

In [3]:
hf_df = pd.read_excel(hf_path)

print(f"HF prioritized variants loaded: {len(hf_df):,} rows")
print(f"Unique rsIDs: {hf_df['rsID'].nunique():,}")

HF prioritized variants loaded: 70,856 rows
Unique rsIDs: 48,390


In [4]:
# Tier 1 and 2 — genome-wide significant and suggestive
sig_df = hf_df[hf_df["p_value"] < 1e-5].copy()
sig_df["significance_tier"] = sig_df["p_value"].apply(
    lambda p: "Genome-wide significant" if p < 5e-8 else "Suggestive"
)

# Tier 3 — best variant per gene for genes with no significant variants
sig_genes = set(sig_df["gene"].unique())

exploratory_df = (
    hf_df[~hf_df["gene"].isin(sig_genes)]
    .dropna(subset=["p_value", "rsID"])
    .sort_values("p_value", ascending=True)
    .drop_duplicates(subset=["gene"])
    .copy()
)
exploratory_df["significance_tier"] = "Exploratory"

# Combine all three tiers
combined_df = pd.concat([sig_df, exploratory_df], ignore_index=True)

# Get unique rsIDs per chromosome
sig_df_unique = (
    combined_df[["rsID", "chromosome", "significance_tier"]]
    .dropna(subset=["rsID", "chromosome"])
    .drop_duplicates(subset=["rsID"])
    .copy()
)

sig_df_unique = sig_df_unique[
    sig_df_unique["rsID"].astype(str).str.startswith("rs")
]

sig_df_unique["chromosome"] = sig_df_unique["chromosome"].astype(int)
chromosomes = sorted(sig_df_unique["chromosome"].unique().tolist())

print(f"Tier 1 (genome-wide significant, p < 5e-8): {(sig_df['significance_tier'] == 'Genome-wide significant').sum()}")
print(f"Tier 2 (suggestive, p < 1e-5):              {(sig_df['significance_tier'] == 'Suggestive').sum()}")
print(f"Tier 3 (exploratory, best per gene):        {len(exploratory_df)}")
print(f"Total unique rsIDs to process:              {len(sig_df_unique)}")
print(f"Chromosomes to process:                     {chromosomes}")
print(f"\nVariants per chromosome:")
print(sig_df_unique["chromosome"].value_counts().sort_index())

Significant variants (p < 1e-5): 20
Unique rsIDs to process: 19
Chromosomes to process: [6, 11, 16, 19]
Variants per chromosome:
chromosome
6     12
11     1
16     1
19     5
Name: count, dtype: int64


In [5]:
def snpclip_batch(rsids, token, population, r2_threshold, maf_threshold,
                  genome_build, retries=3, pause=5):
    payload = json.dumps({
        "snps":          "\n".join(rsids),
        "pop":           population,
        "r2_threshold":  str(r2_threshold),
        "maf_threshold": str(maf_threshold),
        "genome_build":  genome_build,
    })

    for attempt in range(retries):
        try:
            response = requests.post(
                f"{LDLINK_URL}?token={token}",
                headers={"Content-Type": "application/json"},
                data=payload,
                timeout=120,
            )
            if response.status_code == 200:
                return response.text
            else:
                print(f"    HTTP {response.status_code} on attempt {attempt + 1}, retrying...")
                time.sleep(pause * (attempt + 1))
        except requests.exceptions.Timeout:
            print(f"    Timeout on attempt {attempt + 1}, retrying...")
            time.sleep(pause * (attempt + 1))
        except requests.exceptions.RequestException as e:
            print(f"    Request error: {e}, retrying...")
            time.sleep(pause * (attempt + 1))

    print(f"    Failed after {retries} attempts.")
    return None


def parse_snpclip_response(text):
    lines = [l for l in text.strip().split("\n") if l.strip()]
    if not lines:
        return pd.DataFrame()

    rows = []
    for line in lines[1:]:  # skip header
        parts = line.split("\t")
        if len(parts) >= 4:
            rows.append({
                "rsID":     parts[0].strip(),
                "position": parts[1].strip(),
                "alleles":  parts[2].strip(),
                "details":  parts[3].strip(),
            })
        elif len(parts) == 2:
            rows.append({
                "rsID":     parts[0].strip(),
                "position": "",
                "alleles":  "",
                "details":  parts[1].strip(),
            })

    return pd.DataFrame(rows)


# ── Run per chromosome, batching within each chromosome ───────────────────────
all_results = []

for chrom in chromosomes:
    chrom_rsids = (
        sig_df_unique[sig_df_unique["chromosome"] == chrom]["rsID"]
        .tolist()
    )

    if not chrom_rsids:
        continue

    # Split into batches of BATCH_SIZE within this chromosome
    batches   = [chrom_rsids[i:i + BATCH_SIZE] for i in range(0, len(chrom_rsids), BATCH_SIZE)]
    n_batches = len(batches)

    print(f"Chromosome {chrom}: {len(chrom_rsids)} rsIDs, {n_batches} batch(es)")

    for i, batch in enumerate(batches):
        print(f"  Batch {i + 1}/{n_batches} ({len(batch)} SNPs)...", end=" ", flush=True)

        response_text = snpclip_batch(
            rsids         = batch,
            token         = LDLINK_TOKEN,
            population    = POPULATION,
            r2_threshold  = R2_THRESHOLD,
            maf_threshold = MAF_THRESHOLD,
            genome_build  = GENOME_BUILD,
        )

        if response_text:
            batch_df = parse_snpclip_response(response_text)
            all_results.append(batch_df)
            kept = (batch_df["details"] == "Variant kept.").sum() if not batch_df.empty else 0
            print(f"kept {kept} / {len(batch)}")
        else:
            print("failed, skipping batch")

        time.sleep(3)

print("\nDone.")

Chromosome 6: 12 rsIDs, 1 batch(es)
  Batch 1/1 (12 SNPs)... kept 2 / 12
Chromosome 11: 1 rsIDs, 1 batch(es)
  Batch 1/1 (1 SNPs)... kept 0 / 1
Chromosome 16: 1 rsIDs, 1 batch(es)
  Batch 1/1 (1 SNPs)... kept 0 / 1
Chromosome 19: 5 rsIDs, 1 batch(es)
  Batch 1/1 (5 SNPs)... kept 1 / 5

Done.


In [6]:
if not all_results:
    raise RuntimeError("No results returned from SNPclip API.")

snpclip_df = pd.concat(all_results, ignore_index=True)

kept_df    = snpclip_df[snpclip_df["details"] == "Variant kept."].copy()
removed_df = snpclip_df[snpclip_df["details"] != "Variant kept."].copy()
error_df   = snpclip_df[
    ~snpclip_df["details"].str.startswith("Variant kept") &
    ~snpclip_df["details"].str.contains("removed", case=False, na=False)
].copy()

lead_snps = set(kept_df["rsID"].tolist())

print(f"Total SNPs processed:            {len(snpclip_df):,}")
print(f"Kept (independent):              {len(kept_df):,}")
print(f"Removed (correlated or low MAF): {len(removed_df):,}")
print(f"Errors / not found:              {len(error_df):,}")

Total SNPs processed:            19
Kept (independent):              3
Removed (correlated or low MAF): 16
Errors / not found:              0


In [7]:
tier_map = combined_df.drop_duplicates(subset=["rsID"])[["rsID", "significance_tier"]]
hf_lead_df = hf_df[hf_df["rsID"].isin(lead_snps)].copy().reset_index(drop=True)
hf_lead_df = hf_lead_df.merge(tier_map, on="rsID", how="left")

print(f"HF lead SNPs table: {len(hf_lead_df):,} rows, {hf_lead_df.shape[1]} columns")
print(f"Unique lead rsIDs retained: {hf_lead_df['rsID'].nunique():,}")
print(f"\nSignificance tier breakdown:")
print(hf_lead_df["significance_tier"].value_counts())

hf_lead_df.to_excel(out_xlsx, index=False)

HF lead SNPs table: 6 rows, 21 columns
Unique lead rsIDs retained: 3
